# Lab 0 · เรียก LLM ครั้งแรก

Notebook นี้ใช้เวลาราว 15–20 นาที เป้าหมายคือยืนยันว่าเครื่องของคุณเรียก LLM ผ่าน LangChain ได้ และได้เห็นแนวคิดจาก Module 0 ทำงานจริง: message, streaming, temperature, token และ structured output

**ก่อนรัน:** เลือก kernel เป็น `.venv` ของโฟลเดอร์ `agentic-ai` (มุมขวาบนของ notebook) แล้วรันทีละ cell ด้วย `Shift+Enter`

## 1. เลือก model

`get_llm()` อ่านค่าจากไฟล์ `.env` ว่าจะใช้ provider และ model ตัวไหน ทุก lab ในวิชานี้ใช้ function เดียวกันนี้ คุณจึงสลับระหว่าง local model กับ API ได้โดยไม่ต้องแก้ code

In [1]:
from agentlab import get_llm, provider_settings

provider, model, _ = provider_settings()
print(f"provider = {provider}\nmodel    = {model}")

llm = get_llm()

provider = ollama
model    = qwen3.5:4b


## 2. เรียกครั้งแรก

`invoke()` ส่งข้อความไปแล้วรอคำตอบทั้งก้อน สิ่งที่ได้กลับมาคือ message ที่มีทั้งข้อความและจำนวน token ที่ใช้

In [2]:
reply = llm.invoke("อธิบายคำว่า token ในบริบทของ LLM ภายใน 2 ประโยค")

print(reply.text)
print("\nusage:", reply.usage_metadata)

ในบริบทของ Large Language Model (LLM) **Token** คือหน่วยย่อยที่สุดของการประมวลผลภาษา ซึ่งประกอบด้วยตัวอักษร ตัวเลข หรือสัญลักษณ์ที่รวมกลุ่มกันเป็นคำหรือส่วนหนึ่งของคำ เพื่อช่วยให้โมเดลเข้าใจโครงสร้างและบริบทของข้อความได้อย่างมีประสิทธิภาพ การแบ่งข้อความออกเป็น Token ช่วยลดความซับซ้อนในการคำนวณและเพิ่มความเร็วในการเรียนรู้รูปแบบต่างๆ ของภาษา

usage: {'input_tokens': 27, 'output_tokens': 73, 'total_tokens': 100}


## 3. Message และ role

ใน Module 0 เราเห็นว่า API รับรายการ message ที่มี role กำกับ `system` กำหนดกติกา ส่วน `user` คือคำถาม ลองแก้ข้อความใน `system` แล้วดูว่าคำตอบเปลี่ยนอย่างไร

In [3]:
messages = [
    ("system", "You are an accounting assistant. Answer in Thai, in one short sentence."),
    ("user", "ใบกำกับภาษีต่างจากใบเสร็จรับเงินอย่างไร"),
]
print(llm.invoke(messages).text)

ใบกำกับภาษีเป็นเอกสารทางบัญชีสำหรับสินค้าและบริการที่ต้องเสียภาษีมูลค่าเพิ่ม ในขณะที่ใบเสร็จรับเงินเป็นหลักฐานการจ่ายเงินซื้อหรือขายที่ไม่ได้ระบุรายละเอียดภาษีมูลค่าเพิ่ม


## 4. Streaming

Model สร้างคำตอบทีละ token `stream()` ให้เราเห็นแต่ละชิ้นทันทีที่สร้างเสร็จ แทนที่จะรอจนจบ

In [4]:
for chunk in llm.stream("นับเลข 1 ถึง 10 เป็นภาษาไทย คั่นด้วยจุลภาค"):
    print(chunk.text, end="|", flush=True)

๑|.|๒|.|๓|.|๔|.|๕|.|๖|.|๗|.|๘|.|๙|.|๑|๐|||

## 5. Temperature

รัน prompt เดิม 3 ครั้งที่ temperature 0 และอีก 3 ครั้งที่ 1.2 แล้วเทียบว่าชุดไหนคำตอบซ้ำกันมากกว่า

In [5]:
prompt = "ตั้งชื่อร้านกาแฟ 1 ชื่อ ตอบเฉพาะชื่อร้าน"

for temperature in (0, 1.2):
    model_t = get_llm(temperature=temperature)
    names = [model_t.invoke(prompt).text.strip() for _ in range(3)]
    print(f"temperature={temperature}: {names}")

temperature=0: ['คาเฟ่ยามเช้า', 'คาเฟ่ยามเช้า', 'คาเฟ่ยามเช้า']
temperature=1.2: ['บ้านกาแฟ', 'อามิกี้', 'วันเพีย่']


## 6. ภาษาไทยใช้ token มากกว่าจริงหรือไม่

สองประโยคด้านล่างมีความหมายเดียวกัน เราดูจำนวน input token ที่ model รายงานกลับมา ตัวเลขรวม token ของโครง message ไว้ด้วย ซึ่งเท่ากันทั้งสองกรณี ส่วนต่างจึงมาจากตัวประโยค

In [6]:
sentences = {
    "en": "The customer must pay the invoice within thirty days of the delivery date.",
    "th": "ลูกค้าต้องชำระเงินตามใบแจ้งหนี้ภายในสามสิบวันนับจากวันที่ส่งมอบสินค้า",
}
for lang, sentence in sentences.items():
    usage = llm.invoke(sentence + "\n\nReply with: ok").usage_metadata
    print(f"{lang}: {len(sentence):3d} characters -> {usage['input_tokens']} input tokens")

en:  74 characters -> 31 input tokens
th:  69 characters -> 34 input tokens


## 7. Structured output

งาน document intelligence ต้องการข้อมูลที่โปรแกรมนำไปใช้ต่อได้ เรากำหนด schema ด้วย Pydantic แล้วให้ model ตอบตามนั้น นี่คือเทคนิคหลักของ Lab 1

In [7]:
from pydantic import BaseModel, Field


class Invoice(BaseModel):
    invoice_no: str = Field(description="Invoice number exactly as printed")
    seller: str
    total: float = Field(description="Grand total including VAT")
    currency: str = Field(description="ISO currency code such as THB")


text = """
ใบแจ้งหนี้ เลขที่ INV-2026-0042
ผู้ขาย: บริษัท สยามเปเปอร์ จำกัด
กระดาษ A4 80 แกรม 50 รีม @ 110.00 = 5,500.00
ภาษีมูลค่าเพิ่ม 7% = 385.00
รวมทั้งสิ้น 5,885.00 บาท
"""

invoice = llm.with_structured_output(Invoice).invoke(f"Extract the invoice fields.\n{text}")
print(invoice)
print("total is a number:", invoice.total + 1)

invoice_no='INV-2026-0042' seller='บริษัท สยามเปเปอร์ จำกัด' total=5885.0 currency='THB'
total is a number: 5886.0


## 8. สลับ provider

ส่งชื่อ provider ให้ `get_llm()` เพื่อใช้ตัวอื่นชั่วคราว cell นี้ลองทุก provider ที่คุณตั้งค่าไว้ ตัวที่ยังไม่มี key จะถูกข้าม

In [8]:
import time

from agentlab import PROVIDERS

for name in PROVIDERS:
    try:
        start = time.time()
        answer = get_llm(name).invoke("Reply with exactly one word: ready").text.strip()
        print(f"{name:10s} {answer!r} in {time.time() - start:.1f}s")
    except Exception as error:
        print(f"{name:10s} skipped: {str(error)[:90]}")

ollama     'ready' in 5.4s
openrouter skipped: OPENROUTER_API_KEY is empty. Add your key to /Users/preme/agentic-ai/.env (see .env.exampl
groq       skipped: GROQ_API_KEY is empty. Add your key to /Users/preme/agentic-ai/.env (see .env.example).
google     skipped: GOOGLE_API_KEY is empty. Add your key to /Users/preme/agentic-ai/.env (see .env.example).


## ลองทำเอง

1. แก้ `system` message ในข้อ 3 ให้ model ตอบเป็นภาษาอังกฤษแบบ bullet 3 ข้อ
2. เพิ่ม field `vat: float` ใน `Invoice` แล้วรันข้อ 7 ใหม่ model ดึงค่าได้ถูกหรือไม่
3. แก้ยอดรวมในข้อความ invoice ให้ไม่ตรงกับผลบวกของรายการ แล้วถาม model ว่ายอดรวมถูกหรือไม่ จดไว้ว่า model ตอบอย่างไร เราจะกลับมาที่ปัญหานี้ใน Lab 2

**รันเป็นโปรแกรม:** ไฟล์ `hello_llm.py` ในโฟลเดอร์เดียวกันทำสิ่งเดียวกับ notebook นี้ในรูปของโปรแกรมที่เรียกจาก terminal ได้ ดูวิธีรันในคู่มือ Lab 0

**สิ่งที่ต้องส่ง:** ดูขั้นตอนสุดท้ายในคู่มือ Lab 0

1. แก้ `system` message ในข้อ 3 ให้ model ตอบเป็นภาษาอังกฤษแบบ bullet 3 ข้อ

In [9]:
messages_q1 = [
    ("system","You are an accounting assistant. Answer in English as 3 bullet points."),
    ("user", "ใบกำกับภาษีต่างจากใบเสร็จรับเงินอย่างไร"),
]
print(llm.invoke(messages_q1).text)

- **ใบกำกับภาษี** เป็นเอกสารทางกฎหมายที่ออกโดยผู้ขายเพื่อแสดงรายละเอียดสินค้า/บริการและยอดภาษีมูลค่าเพิ่ม (VAT) ที่ต้องเสียจ่ายให้รัฐ เพื่อใช้ประกอบการยื่นรายงานภาษีของทั้งผู้ซื้อและผู้ขาย
- **ใบเสร็จรับเงิน** เป็นหลักฐานการชำระเงินจากธุรกิจทั่วไปที่ไม่ได้จดทะเบียนหรือไม่ได้เสียภาษี VAT ซึ่งไม่มีการระบุข้อมูลเกี่ยวกับภาษีมูลค่าเพิ่ม
- **หน้าที่ทางบัญชี** ใบกำกับภาษีถูกนำไปคำนวณเพื่อหักลดหย่อนภาษี (Input Tax) ของบริษัท ในขณะที่ใบเสร็จรับเงินใช้เพียงเป็นหลักฐานเบิกจ่ายค่าใช้จ่ายและบันทึกบัญชีตามหลักเกณฑ์การ deductible expense


- ผลลัพธ์: โมเดลตอบเป็น bullet 3 ข้อถูกต้อง แต่ตอบเป็น ภาษาไทย ปนคำศัพท์ภาษาอังกฤษ
- เหตุผล: โมเดลขนาดเล็ก (4B Parameter) มีแนวโน้มที่จะให้น้ำหนักกับภาษาของคำถามฝั่ง user มากกว่าคำสั่งใน system เมื่อเจอคำถามเป็นภาษาไทย จึงเผลอตอบกลับมาเป็นภาษาไทยตามคำถาม

2. เพิ่ม field `vat: float` ใน `Invoice` แล้วรันข้อ 7 ใหม่ model ดึงค่าได้ถูกหรือไม่

In [12]:
from pydantic import BaseModel, Field


class Invoice(BaseModel):
    invoice_no: str = Field(description="Invoice number exactly as printed")
    seller: str
    vat: float
    total: float = Field(description="Grand total including VAT")
    currency: str = Field(description="ISO currency code such as THB")


text = """
ใบแจ้งหนี้ เลขที่ INV-2026-0042
ผู้ขาย: บริษัท สยามเปเปอร์ จำกัด
กระดาษ A4 80 แกรม 50 รีม @ 110.00 = 5,500.00
ภาษีมูลค่าเพิ่ม 7% = 385.00
รวมทั้งสิ้น 5,885.00 บาท
"""

invoice = llm.with_structured_output(Invoice).invoke(f"Extract the invoice fields.\n{text}")
print(invoice)
print("total is a number:", invoice.total + 1)

invoice_no='INV-2026-0042' seller='บริษัท สยามเปเปอร์ จำกัด' vat=7.0 total=5885.0 currency='THB'
total is a number: 5886.0


- ผลลัพธ์: โมเดลคืนค่าเป็น vat=7.0 โมเดลดึงค่าไม่ถูกต้อง
- เหตุผล: เพราะประกาศแค่ vat: float โดยไม่มีคำอธิบายหรือคำสั่งที่เจาะจง โมเดลจึงเลือกหยิบเลข 7.0 (จาก 7%) มาใส่ แทนที่จะเป็นยอดเงิน 385.00

3. แก้ยอดรวมในข้อความ invoice ให้ไม่ตรงกับผลบวกของรายการ แล้วถาม model ว่ายอดรวมถูกหรือไม่

In [11]:
wrong_invoice_text = """
ใบแจ้งหนี้ เลขที่ INV-2026-0042
ผู้ขาย: บริษัท สยามเปเปอร์ จำกัด
กระดาษ A4 80 แกรม 50 รีม @ 110.00 = 5,500.00
ภาษีมูลค่าเพิ่ม 7% = 385.00
รวมทั้งสิ้น 9,999.00 บาท
"""

prompt_q3 = f"""จากรายการใบแจ้งหนี้นี้ ยอดรวมทั้งสิ้นถูกต้องตามผลบวกของรายการสินค้าและภาษีหรือไม่? จงคำนวณและตอบว่าถูกหรือผิด
{wrong_invoice_text}"""

reply_q3 = llm.invoke(prompt_q3)
print(reply_q3.text)

จากการตรวจสอบข้อมูลในใบแจ้งหนี้ของคุณ พบว่า **ยอดรวมไม่ถูกต้อง** ครับ

เหตุผลคือ การคำนวณภาษีมูลค่าเพิ่ม (VAT) และยอดรวมทั้งสิ้นจากจำนวนสินค้ามีตัวเลขที่ขัดแย้งกันเมื่อทำการบวกจริง ดังนี้:

1.  **ตรวจสอบยอดสินค้า:**
    *   จำนวนกระดาษ A4 80 แกรม = 50 รีม
    *   ราคาต่อหน่วย = 110.00 บาท
    *   ยอดรวมสินค้า (ก่อนภาษี) = $50 \times 110.00 = 5,500.00$ บาท **(ถูกต้อง)**

2.  **ตรวจสอบยอดภาษีมูลค่าเพิ่ม (7%):**
    *   คำนวณจากยอดสินค้า: $5,500.00 \times 0.07 = 385.00$ บาท **(ถูกต้อง)**
    *   *หมายเหตุ:* ในใบแจ้งหนี้ของคุณระบุไว้ว่า "ภาษีมูลค่าเพิ่ม 7% = 385.00" ซึ่งตัวเลขนี้คำนวณได้ถูกต้องตามยอดสินค้า

3.  **ตรวจสอบยอดรวมทั้งสิ้น (Total):**
    *   ยอดรวมควรจะเป็นผลบวกของ "ยอดสินค้า" + "ภาษีมูลค่าเพิ่ม"
    *   การคำนวณที่ถูกต้อง: $5,500.00 + 385.00 = \mathbf{5,885.00}$ บาท
    *   **แต่ในใบแจ้งหนี้ระบุไว้ว่า:** "รวมทั้งสิ้น 9,999.00 บาท"

**สรุปผลการคำนวณ:**
*   ยอดที่ควรจะเป็น: 5,885.00 บาท
*   ยอดที่ระบุในเอกสาร: 9,999.00 บาท
*   ความแตกต่าง: 4,114.00 บาท (ตัวเลข 9,999.00 ไม่สามารถ

ผลลัพธ์: โมเดลตอบว่า "ยอดรวมไม่ถูกต้อง" และแสดงวิธีคิดเปรียบเทียบว่ายอดที่ถูกต้องควรเป็น 5,885.00 แต่ในข้อความinvoiceระบุ 9,999.00 จึงตอบว่า ผิด